# Exercice 07 - Ingestion PostgreSQL vers Bronze

## Objectifs
- Extraire des donnees de PostgreSQL
- Sauvegarder dans la couche Bronze de MinIO
- Organiser les donnees par date d'ingestion
- Creer un pipeline d'ingestion reutilisable

---

## 1. Architecture d'ingestion

```
+----------------+                    +------------------------+
|                |                    |        MinIO           |
|   PostgreSQL   |     SPARK          |                        |
|                |  =============>    |  +------------------+  |
|  +----------+  |                    |  |     BRONZE       |  |
|  | customers|  |                    |  +------------------+  |
|  | products |  |                    |  | /customers/      |  |
|  | orders   |  |                    |  |   /2024-01-15/   |  |
|  | ...      |  |                    |  | /products/       |  |
|  +----------+  |                    |  |   /2024-01-15/   |  |
|                |                    |  | /orders/         |  |
+----------------+                    |  |   /2024-01-15/   |  |
                                      |  +------------------+  |
                                      +------------------------+

Format Bronze : donnees brutes en Parquet
Organisation  : /table/date_ingestion/
```

## 2. Configuration Spark pour PostgreSQL et MinIO

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime

# Creer la SparkSession avec les configurations
spark = SparkSession.builder \
    .appName("Ingestion PostgreSQL vers Bronze") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark configure pour PostgreSQL et MinIO")

Spark configure pour PostgreSQL et MinIO


In [4]:
# Configuration PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

# Date d'ingestion pour organiser les fichiers
date_ingestion = datetime.now().strftime("%Y-%m-%d")
print(f"Date d'ingestion : {date_ingestion}")

Date d'ingestion : 2026-03-25


## 3. Ingerer une table simple

In [5]:
# Lire la table customers depuis PostgreSQL
df_customers = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=jdbc_properties
)

print(f"Customers : {df_customers.count()} lignes")
df_customers.show(5)

Customers : 91 lignes
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|     Berlin|  NULL|      12209|Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...|México D.F.|  NULL|      05021| Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312|México D.F.|  NULL|      05023| Mexico|  (5) 555-3932|    

In [6]:
# Sauvegarder dans Bronze
chemin_bronze = f"s3a://bronze/customers/{date_ingestion}"

df_customers.write \
    .mode("overwrite") \
    .parquet(chemin_bronze)

print(f"Sauvegarde reussie : {chemin_bronze}")

Sauvegarde reussie : s3a://bronze/customers/2026-03-25


In [7]:
# Verifier la sauvegarde
df_check = spark.read.parquet(chemin_bronze)
print(f"Verification : {df_check.count()} lignes lues depuis Bronze")

Verification : 91 lignes lues depuis Bronze


## 4. Fonction d'ingestion reutilisable

In [8]:
def ingerer_table(nom_table, date=None):
    """
    Ingere une table PostgreSQL vers le bucket Bronze.
    
    Args:
        nom_table: Nom de la table a ingerer
        date: Date d'ingestion (defaut: aujourd'hui)
    
    Returns:
        Nombre de lignes ingerees
    """
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    
    # Lire depuis PostgreSQL
    df = spark.read.jdbc(
        url="jdbc:postgresql://postgres:5432/app",
        table=nom_table,
        properties={
            "user": "postgres",
            "password": "postgres",
            "driver": "org.postgresql.Driver"
        }
    )
    
    # Sauvegarder dans Bronze
    chemin = f"s3a://bronze/{nom_table}/{date}"
    df.write.mode("overwrite").parquet(chemin)
    
    nb_lignes = df.count()
    print(f"[OK] {nom_table} : {nb_lignes} lignes -> {chemin}")
    
    return nb_lignes

In [9]:
# Tester la fonction
ingerer_table("products")

[OK] products : 77 lignes -> s3a://bronze/products/2026-03-25


77

## 5. Ingerer toutes les tables Northwind

In [10]:
# Liste des tables a ingerer
tables_northwind = [
    "categories",
    "customers",
    "employees",
    "orders",
    "order_details",
    "products",
    "shippers",
    "suppliers",
    "region",
    "territories"
]

print(f"Tables a ingerer : {len(tables_northwind)}")

Tables a ingerer : 10


In [11]:
# Ingerer toutes les tables
resultats = {}

print("=" * 50)
print("INGESTION NORTHWIND VERS BRONZE")
print("=" * 50)

for table in tables_northwind:
    try:
        nb_lignes = ingerer_table(table, date_ingestion)
        resultats[table] = {"status": "OK", "lignes": nb_lignes}
    except Exception as e:
        print(f"[ERREUR] {table} : {e}")
        resultats[table] = {"status": "ERREUR", "erreur": str(e)}

print("=" * 50)
print("INGESTION TERMINEE")
print("=" * 50)

INGESTION NORTHWIND VERS BRONZE
[OK] categories : 8 lignes -> s3a://bronze/categories/2026-03-25
[OK] customers : 91 lignes -> s3a://bronze/customers/2026-03-25
[OK] employees : 9 lignes -> s3a://bronze/employees/2026-03-25
[OK] orders : 830 lignes -> s3a://bronze/orders/2026-03-25
[OK] order_details : 2155 lignes -> s3a://bronze/order_details/2026-03-25
[OK] products : 77 lignes -> s3a://bronze/products/2026-03-25
[OK] shippers : 6 lignes -> s3a://bronze/shippers/2026-03-25
[OK] suppliers : 29 lignes -> s3a://bronze/suppliers/2026-03-25
[OK] region : 4 lignes -> s3a://bronze/region/2026-03-25
[OK] territories : 53 lignes -> s3a://bronze/territories/2026-03-25
INGESTION TERMINEE


In [12]:
# Resume de l'ingestion
print("\nResume :")
total_lignes = 0
for table, info in resultats.items():
    if info["status"] == "OK":
        print(f"  {table}: {info['lignes']} lignes")
        total_lignes += info["lignes"]
    else:
        print(f"  {table}: ERREUR")

print(f"\nTotal : {total_lignes} lignes ingerees")


Resume :
  categories: 8 lignes
  customers: 91 lignes
  employees: 9 lignes
  orders: 830 lignes
  order_details: 2155 lignes
  products: 77 lignes
  shippers: 6 lignes
  suppliers: 29 lignes
  region: 4 lignes
  territories: 53 lignes

Total : 3262 lignes ingerees


## 6. Ajouter des metadonnees d'ingestion

In [13]:
from pyspark.sql import functions as F

def ingerer_table_avec_metadata(nom_table, date=None):
    """
    Ingere une table avec des colonnes de metadata.
    """
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    
    timestamp_ingestion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Lire depuis PostgreSQL
    df = spark.read.jdbc(
        url="jdbc:postgresql://postgres:5432/app",
        table=nom_table,
        properties={
            "user": "postgres",
            "password": "postgres",
            "driver": "org.postgresql.Driver"
        }
    )
    
    # Ajouter les metadonnees
    df = df.withColumn("_source", F.lit("postgresql")) \
           .withColumn("_table", F.lit(nom_table)) \
           .withColumn("_ingestion_date", F.lit(date)) \
           .withColumn("_ingestion_timestamp", F.lit(timestamp_ingestion))
    
    # Sauvegarder dans Bronze
    chemin = f"s3a://bronze/{nom_table}/{date}"
    df.write.mode("overwrite").parquet(chemin)
    
    nb_lignes = df.count()
    print(f"[OK] {nom_table} : {nb_lignes} lignes (avec metadata) -> {chemin}")
    
    return nb_lignes

In [14]:
# Tester avec metadata
ingerer_table_avec_metadata("categories")

# Verifier les metadonnees
df_test = spark.read.parquet(f"s3a://bronze/categories/{date_ingestion}")
df_test.select("category_id", "category_name", "_source", "_table", "_ingestion_date").show()

[OK] categories : 8 lignes (avec metadata) -> s3a://bronze/categories/2026-03-25
+-----------+--------------+----------+----------+---------------+
|category_id| category_name|   _source|    _table|_ingestion_date|
+-----------+--------------+----------+----------+---------------+
|          1|     Beverages|postgresql|categories|     2026-03-25|
|          2|    Condiments|postgresql|categories|     2026-03-25|
|          3|   Confections|postgresql|categories|     2026-03-25|
|          4|Dairy Products|postgresql|categories|     2026-03-25|
|          5|Grains/Cereals|postgresql|categories|     2026-03-25|
|          6|  Meat/Poultry|postgresql|categories|     2026-03-25|
|          7|       Produce|postgresql|categories|     2026-03-25|
|          8|       Seafood|postgresql|categories|     2026-03-25|
+-----------+--------------+----------+----------+---------------+



## 7. Verifier le contenu de Bronze

In [15]:
# Lister les fichiers dans Bronze
print("Contenu du bucket Bronze :")
print("=" * 50)

for table in tables_northwind:
    try:
        chemin = f"s3a://bronze/{table}/{date_ingestion}"
        df = spark.read.parquet(chemin)
        print(f"{table}: {df.count()} lignes")
    except:
        print(f"{table}: non trouve")

Contenu du bucket Bronze :
categories: 8 lignes
customers: 91 lignes
employees: 9 lignes
orders: 830 lignes
order_details: 2155 lignes
products: 77 lignes
shippers: 6 lignes
suppliers: 29 lignes
region: 4 lignes
territories: 53 lignes


---

## Exercice

**Objectif** : Creer un script d'ingestion complet

**Consigne** :
1. Creez une fonction `ingestion_complete()` qui :
   - Ingere toutes les tables avec metadonnees
   - Affiche un rapport final
   - Retourne un dictionnaire avec les statistiques

2. Ajoutez une colonne `_nb_colonnes` aux metadonnees

A vous de jouer :

In [18]:
def ingestion_complete():
    """
    Ingere toutes les tables Northwind vers Bronze avec metadata.
    
    Returns:
        dict: Statistiques d'ingestion avec status et nombre de lignes par table
    """
    from pyspark.sql import functions as F
    
    # ===== ETAPE 1: Preparer les variables =====
    # Dictionnaire pour stocker les resultats
    resultats = {}
    
    # Date et heure d'ingestion
    date_ingestion = datetime.now().strftime("%Y-%m-%d")
    timestamp_ingestion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Configuration PostgreSQL
    jdbc_url = "jdbc:postgresql://postgres:5432/app"
    jdbc_properties = {
        "user": "postgres",
        "password": "postgres",
        "driver": "org.postgresql.Driver"
    }
    
    # ===== ETAPE 2: Afficher l'entete =====
    print("\n" + "=" * 70)
    print("INGESTION COMPLETE NORTHWIND VERS BRONZE")
    print("=" * 70)
    print(f"Date d'ingestion: {date_ingestion}")
    print(f"Heure: {timestamp_ingestion}")
    print("=" * 70 + "\n")
    
    # ===== ETAPE 3: Boucler sur chaque table =====
    for nom_table in tables_northwind:
        try:
            # 3a. Lire la table depuis PostgreSQL
            df = spark.read.jdbc(
                url=jdbc_url,
                table=nom_table,
                properties=jdbc_properties
            )
            
            # 3b. Compter le nombre de lignes et de colonnes
            nb_lignes = df.count()
            nb_colonnes = len(df.columns)
            
            # 3c. Ajouter les colonnes de metadata
            df = df.withColumn("_source", F.lit("postgresql")) \
                   .withColumn("_table", F.lit(nom_table)) \
                   .withColumn("_ingestion_date", F.lit(date_ingestion)) \
                   .withColumn("_ingestion_timestamp", F.lit(timestamp_ingestion)) \
                   .withColumn("_nb_colonnes", F.lit(nb_colonnes))
            
            # 3d. Definir le chemin de sauvegarde
            chemin_bronze = f"s3a://bronze/{nom_table}/{date_ingestion}"
            
            # 3e. Sauvegarder dans MinIO
            df.write.mode("overwrite").parquet(chemin_bronze)
            
            # 3f. Enregistrer le succes
            resultats[nom_table] = {
                "status": "OK",
                "lignes": nb_lignes,
                "colonnes": nb_colonnes,
                "chemin": chemin_bronze
            }
            
            # Afficher le progres
            print(f"✓ {nom_table:20} | {nb_lignes:6} lignes | {nb_colonnes:3} colonnes")
            
        except Exception as e:
            # 3g. Enregistrer l'erreur
            resultats[nom_table] = {
                "status": "ERREUR",
                "erreur": str(e)
            }
            
            # Afficher l'erreur
            print(f"✗ {nom_table:20} | ERREUR: {str(e)[:40]}")
    
    # ===== ETAPE 4: Afficher le rapport final =====
    print("\n" + "=" * 70)
    print("RAPPORT FINAL")
    print("=" * 70)
    
    # Compter les succes et erreurs
    succes = sum(1 for r in resultats.values() if r["status"] == "OK")
    erreurs = sum(1 for r in resultats.values() if r["status"] == "ERREUR")
    total_lignes = sum(r.get("lignes", 0) for r in resultats.values() if r["status"] == "OK")
    
    print(f"\nTable ingerees   : {succes}/{len(tables_northwind)}")
    print(f"Tables en erreur         : {erreurs}/{len(tables_northwind)}")
    print(f"Total de lignes          : {total_lignes:,}")
    
    # Afficher les details
    print("\nDETAILS PAR TABLE :")
    print("-" * 70)
    for table, info in sorted(resultats.items()):
        if info["status"] == "OK":
            print(f"  {table:20} | OK      | {info['lignes']:6} lignes | {info['colonnes']:2} col")
        else:
            print(f"  {table:20} | ERREUR  | {info['erreur'][:35]}")
    
    print("=" * 70 + "\n")
    
    # ===== ETAPE 5: Retourner les resultats =====
    return resultats

In [19]:
# Executer la fonction d'ingestion complete
stats = ingestion_complete()

# Afficher les statistiques brutes (pour verification)
print("\nDictionnaire des statistiques retourne :")
print(stats)


INGESTION COMPLETE NORTHWIND VERS BRONZE
Date d'ingestion: 2026-03-25
Heure: 2026-03-25 12:46:51

✓ categories           |      8 lignes |   4 colonnes
✓ customers            |     91 lignes |  11 colonnes
✓ employees            |      9 lignes |  18 colonnes
✓ orders               |    830 lignes |  14 colonnes
✓ order_details        |   2155 lignes |   5 colonnes
✓ products             |     77 lignes |  10 colonnes
✓ shippers             |      6 lignes |   3 colonnes
✓ suppliers            |     29 lignes |  12 colonnes
✓ region               |      4 lignes |   2 colonnes
✓ territories          |     53 lignes |   3 colonnes

RAPPORT FINAL

Table ingerees   : 10/10
Tables en erreur         : 0/10
Total de lignes          : 3,262

DETAILS PAR TABLE :
----------------------------------------------------------------------
  categories           | OK      |      8 lignes |  4 col
  customers            | OK      |     91 lignes | 11 col
  employees            | OK      |      9 ligne

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **extraire des donnees** de PostgreSQL avec Spark
- Comment **sauvegarder** les donnees dans MinIO (Bronze)
- Comment **organiser** les donnees par date d'ingestion
- Comment **ajouter des metadonnees** pour la tracabilite
- Comment creer un **pipeline d'ingestion** reutilisable

### Prochaine etape
Dans le prochain notebook, nous apprendrons a ingerer des donnees depuis le Web (API REST).